# MorphIQ SDK for Python — Option pricing

Value a call or put, compare assumptions, and examine sensitivity to the underlying price.

**Run All once to open the workspace**, then use the controls below. Yahoo prices load automatically on setup and when the ticker or interval changes.
Press Enter or leave the ticker field to apply an edit. You can also enter a price manually. Choose European or American
exercise, the pricing model, strike, time to expiry, volatility, interest and dividend yield.

**Price option** makes one API request. **Price + Greeks** returns the price and all
ten Greeks in one request using the selected option inputs. **Solve IV** uses an entered option premium to solve volatility in one request.
**Analyze premium** solves IV and evaluates price and all ten Greeks at that IV in one request.
**Spot sensitivity** submits nine scenarios in one batch request across ±20% of your underlying price, holding the other inputs fixed.
Each action may consume API credits. Changing controls and loading Yahoo data make
no MorphIQ requests. Completed valuations stay in the comparison table for this session.

Docker includes everything required. Outside Docker, install this repository with
`pip install -e '.[notebooks]'`. Set `MORPHIQ_API_KEY` before starting, or enter it
at the masked prompt when running the workspace. [Get API access](https://developers.morphiqlabs.com).

[Back to start](../start-here.ipynb)


In [ ]:
import math
import os
from datetime import date, timedelta
from getpass import getpass

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
from IPython.display import display

from morphiq_sdk import (
    AnalyticsRequest,
    APIError,
    BatchPriceRequest,
    GreeksRequest,
    ImpliedVolatilityRequest,
    MorphIQ,
    PriceRequest,
    RateLimitError,
    ResponseError,
    TransportError,
)

if not os.environ.get("MORPHIQ_API_KEY"):
    os.environ["MORPHIQ_API_KEY"] = getpass("MorphIQ API key: ")

style = {"description_width": "140px"}
field_layout = widgets.Layout(width="350px")
ticker = widgets.Text(
    value="SPY",
    description="Underlying ticker",
    continuous_update=False,
    style=style,
    layout=field_layout,
)
price_interval = widgets.Dropdown(
    options=[
        ("Daily close", "1d"),
        ("1 minute", "1m"),
        ("5 minutes", "5m"),
        ("15 minutes", "15m"),
        ("30 minutes", "30m"),
        ("1 hour", "60m"),
    ],
    value="1d",
    description="Price interval",
    style=style,
    layout=field_layout,
)
market_output = widgets.Output()
exercise = widgets.Dropdown(
    options=[("European", "european"), ("American", "american")],
    description="Exercise style",
    style=style,
    layout=field_layout,
)
option_type = widgets.Dropdown(
    options=[("Call", "call"), ("Put", "put")],
    description="Option type",
    style=style,
    layout=field_layout,
)
model = widgets.Dropdown(
    options=[("Automatic", "auto"), ("Black–Scholes–Merton", "black_scholes_merton")],
    description="Pricing model",
    style=style,
    layout=field_layout,
)
spot = widgets.FloatText(
    value=100.0, description="Underlying price", style=style, layout=field_layout
)
strike = widgets.FloatText(value=100.0, description="Strike", style=style, layout=field_layout)
expiry = widgets.FloatText(
    value=1.0, description="Expiry (years)", style=style, layout=field_layout
)
volatility = widgets.FloatText(
    value=20.0, description="Volatility (%)", style=style, layout=field_layout
)
rate = widgets.FloatText(
    value=5.0, description="Risk-free rate (%)", style=style, layout=field_layout
)
yield_input = widgets.FloatText(
    value=0.0, description="Dividend yield (%)", style=style, layout=field_layout
)
price_button = widgets.Button(
    description="Price option · 1 request",
    button_style="primary",
    layout=widgets.Layout(width="250px"),
)
sweep_button = widgets.Button(
    description="Spot sensitivity · 1 batch", layout=widgets.Layout(width="280px")
)
greeks_button = widgets.Button(
    description="Price + Greeks · 1 request", layout=widgets.Layout(width="280px")
)
market_price = widgets.FloatText(
    value=10.0, description="Option premium", style=style, layout=field_layout
)
iv_button = widgets.Button(description="Solve IV · 1 request", layout=widgets.Layout(width="250px"))
analytics_button = widgets.Button(
    description="Analyze premium · 1 request", layout=widgets.Layout(width="280px")
)
analytics_output = widgets.Output()
analytics_status = widgets.Label()
last_analytics_response = None
analytics_snapshot = None
analytics_table = None
# Retain the ledger when setup is rerun in this kernel. A new kernel starts a new session.
charge_history = globals().get("charge_history", [])
usage_summary = widgets.Label()
usage_output = widgets.Output()
iv_output = widgets.Output()
iv_status = widgets.Label()
last_iv_response = None
iv_snapshot = None
greeks_output = widgets.Output()
greeks_status = widgets.Label()
last_greeks_response = None
greeks_snapshot = None
greeks_table = None
result_output = widgets.Output()
sweep_output = widgets.Output()
price_history = []
sweep_results = []
last_response = None
spot_source = "Manual input"
controls = [
    ticker,
    price_interval,
    exercise,
    option_type,
    model,
    spot,
    strike,
    expiry,
    volatility,
    rate,
    yield_input,
    price_button,
    greeks_button,
    market_price,
    iv_button,
    analytics_button,
    sweep_button,
]


def update_models(change):
    model.options = (
        [("Automatic", "auto"), ("Black–Scholes–Merton", "black_scholes_merton")]
        if exercise.value == "european"
        else [
            ("Automatic", "auto"),
            ("Bjerksund–Stensland 2002", "bjerksund_stensland_2002"),
            ("Bjerksund–Stensland 1993", "bjerksund_stensland_1993"),
        ]
    )
    model.value = "auto"


def mark_manual(change):
    global spot_source
    spot_source = "Manual input"


exercise.observe(update_models, names="value")
spot.observe(mark_manual, names="value")


def yahoo_price_options(interval):
    if interval == "1d":
        end = date.today()
        return {"interval": interval, "start": end - timedelta(days=14), "end": end}
    return {"interval": interval, "period": "5d"}


def use_yahoo_close(history, symbol, interval="1d"):
    global spot_source
    if history is None or history.empty:
        raise ValueError(
            "Yahoo returned no price history. Check the ticker or enter a price manually."
        )
    value = float(history["Close"].iloc[-1])
    if not math.isfinite(value) or value <= 0:
        raise ValueError("The latest Yahoo close is missing or invalid; enter a price manually.")
    spot.value = value
    timestamp = history.index[-1]
    if interval == "1d":
        spot_source = f"{symbol}: Yahoo daily close on {timestamp.date()}"
        print(f"{spot_source}: {value:.6g}. Completed daily close.")
    else:
        zone = str(timestamp.tz) if timestamp.tz is not None else "timezone not reported"
        spot_source = f"{symbol}: Yahoo {interval} bar at {timestamp.isoformat()} ({zone})"
        print(f"{spot_source}: {value:.6g}.")
        print("Latest returned bar; it may still be forming and Yahoo data may be delayed.")
    print("Strike, volatility, rates and yield remain your assumptions.")


def report_error(error):
    if isinstance(error, RateLimitError):
        print("Rate limited (429). No retry was made.")
        print(
            "Retry-After:", error.retry_after if error.retry_after is not None else "not supplied"
        )
        print("Retry guidance in seconds:", error.retry_delay())
    if isinstance(error, APIError):
        charge = error.metadata.fcu_charged if error.metadata else None
        print(f"HTTP {error.status_code}: {error.code}: {error.message}")
        print(
            f"Request ID: {error.request_id}; "
            f"CU charged: {charge if charge is not None else 'not reported'}"
        )
    elif isinstance(error, (TransportError, ResponseError)):
        print("No usable response. Execution and charges may be unknown; no retry was made.")
        if isinstance(error, ResponseError):
            print("Request ID:", error.request_id)
    else:
        print(str(error))


def load_close(_):
    with market_output:
        market_output.clear_output(wait=True)
        set_busy(True)
        try:
            symbol = ticker.value.strip().upper()
            if not symbol:
                raise ValueError("Enter an underlying ticker.")
            selected_interval = price_interval.value
            history = yf.download(
                symbol,
                **yahoo_price_options(selected_interval),
                prepost=False,
                ignore_tz=False,
                auto_adjust=False,
                keepna=False,  # Use yfinance's standard empty-bar cleanup.
                multi_level_index=False,
                progress=False,
                threads=False,
            )
            use_yahoo_close(history, symbol, selected_interval)
        except Exception as error:
            spot.value = 0.0
            print("Yahoo lookup failed:", error)
            print("Enter an underlying price manually or change the ticker/interval to try again.")
        finally:
            set_busy(False)


def option_inputs():
    values = [
        spot.value,
        strike.value,
        expiry.value,
        rate.value,
        yield_input.value,
    ]
    if not all(math.isfinite(value) for value in values):
        raise ValueError("All inputs must be finite numbers.")
    if min(spot.value, strike.value, expiry.value) <= 0:
        raise ValueError("Underlying price, strike and expiry must be positive.")
    if yield_input.value < 0:
        raise ValueError("Dividend yield must be nonnegative.")
    return dict(
        exercise_style=exercise.value,
        option_type=option_type.value,
        spot=spot.value,
        strike=strike.value,
        time_to_expiry_years=expiry.value,
        risk_free_rate=rate.value / 100,
        dividend_yield=yield_input.value / 100,
        pricing_model=model.value,
    )


def current_request():
    if not math.isfinite(volatility.value) or volatility.value <= 0:
        raise ValueError("Volatility must be finite and positive.")
    return PriceRequest(**option_inputs(), volatility=volatility.value / 100)


def current_iv_request():
    if not math.isfinite(market_price.value) or market_price.value < 0:
        raise ValueError("Option premium must be finite and nonnegative, per share.")
    return ImpliedVolatilityRequest(**option_inputs(), market_price=market_price.value)


def result_row(request, response):
    return {
        "Style": request.exercise_style,
        "Type": request.option_type,
        "Spot": request.spot,
        "Strike": request.strike,
        "Years": request.time_to_expiry_years,
        "Vol (%)": request.volatility * 100,
        "Rate (%)": request.risk_free_rate * 100,
        "Yield (%)": request.dividend_yield * 100,
        "Price": response.result.price,
        "Model": response.result.pricing_model,
        "Version": response.result.model_version,
        "CU charged": response.metadata.fcu_charged
        if response.metadata.fcu_charged is not None
        else "not reported",
        "Request ID": response.metadata.request_id,
    }


def set_busy(busy):
    for control in controls:
        control.disabled = busy


def show_usage():
    reported = sum(row["CU charged"] for row in charge_history if row["CU charged"] is not None)
    unknown = sum(row["CU charged"] is None for row in charge_history)
    usage_summary.value = (
        f"Kernel session: {reported} CU reported; total incomplete ({unknown} unknown charges)."
        if unknown
        else f"Kernel session total: {reported} CU."
    ) + f" Requests: {len(charge_history)}."
    usage_output.outputs = ()
    if charge_history:
        # Keep integers exact, including values above floating-point precision.
        rows = [
            {
                **row,
                "CU charged": str(row["CU charged"])
                if row["CU charged"] is not None
                else "unknown",
            }
            for row in charge_history
        ]
        usage_output.append_display_data(pd.DataFrame(rows))


def tracked_request(action, operation, request):
    try:
        response = operation(request)
    except (APIError, ResponseError, TransportError, KeyboardInterrupt) as error:
        metadata = getattr(error, "metadata", None)
        charge_history.append(
            {
                "Attempt": len(charge_history) + 1,
                "Action": action,
                "Outcome": "failed" if isinstance(error, APIError) else "unknown",
                "Request ID": getattr(error, "request_id", None),
                "CU charged": metadata.fcu_charged if metadata else None,
            }
        )
        show_usage()
        raise
    charge_history.append(
        {
            "Attempt": len(charge_history) + 1,
            "Action": action,
            "Outcome": "success",
            "Request ID": response.metadata.request_id,
            "CU charged": response.metadata.fcu_charged,
        }
    )
    show_usage()
    return response


def price_selected(_):
    global last_response
    last_response = None
    with result_output:
        result_output.clear_output(wait=True)
        set_busy(True)
        try:
            request = current_request()
            with MorphIQ() as client:
                response = tracked_request("Price option", client.price, request)
            last_response = response
            price_history.append(
                {"Underlying source": spot_source, **result_row(request, response)}
            )
            print(f"Option value: {response.result.price:.8g} per share, in the input currency.")
            print("No contract multiplier applied. Completed valuations:")
            display(pd.DataFrame(price_history))
        except (APIError, TransportError, ResponseError, ValueError) as error:
            report_error(error)
        finally:
            set_busy(False)


def calculate_greeks(_):
    global last_greeks_response, greeks_snapshot, greeks_table
    set_busy(True)
    greeks_status.value = "Calculating price and Greeks…"
    last_greeks_response = greeks_snapshot = greeks_table = None
    greeks_output.outputs = ()
    try:
        request = GreeksRequest(**current_request().to_wire())
        with MorphIQ() as client:
            response = tracked_request("Price + Greeks", client.greeks, request)
        last_greeks_response = response
        greeks_snapshot = {"Underlying source": spot_source, **result_row(request, response)}
        units = {
            "delta": "dV/dS",
            "gamma": "d²V/dS²",
            "theta": "dV/dt · per year",
            "vega": "dV/dσ · per unit volatility",
            "rho": "dV/dr · per unit rate",
            "vanna": "d²V/(dS dσ) · per unit volatility",
            "volga": "d²V/dσ² · per unit volatility squared",
            "charm": "d²V/(dS dt) · per year",
            "veta": "d²V/(dσ dt) · per year per unit volatility",
            "color": "d³V/(dS² dt) · per year",
        }
        greeks_table = pd.DataFrame(
            [
                {"Greek": name, "Value": getattr(response.result.greeks, name), "Basis": basis}
                for name, basis in units.items()
            ]
        ).set_index("Greek")
        with greeks_output.hold_sync():
            greeks_output.append_stdout(
                f"Option value: {response.result.price:.12g} per share.\n"
                "Greeks use the API's raw units; no calendar-day, percentage-point or "
                "contract-multiplier conversion is applied.\n"
            )
            greeks_output.append_display_data(greeks_table.style.format({"Value": "{:.12g}"}))
            greeks_output.append_display_data(pd.DataFrame([greeks_snapshot]))
            greeks_output.append_stdout(
                "Engines: "
                + "; ".join(
                    f"{e.name} {e.version} ({e.revision})" for e in response.metadata.engines
                )
                + "\n"
            )
        greeks_status.value = "Greeks complete. Results retain the inputs shown below."
    except APIError as error:
        if isinstance(error, RateLimitError):
            greeks_output.append_stdout(
                "Rate limited (429). No retry was made. Retry-After: "
                + (error.retry_after if error.retry_after is not None else "not supplied")
                + "\n"
            )
        charge = error.metadata.fcu_charged if error.metadata else None
        greeks_output.append_stdout(
            f"HTTP {error.status_code}: {error.code}: {error.message}\n"
            f"Request ID: {error.request_id}; "
            f"CU charged: {charge if charge is not None else 'not reported'}\n"
        )
        greeks_status.value = "Request failed; no automatic retry."
    except (TransportError, ResponseError) as error:
        greeks_output.append_stdout(
            "No usable response. Execution and charges may be unknown; no retry was made.\n"
        )
        if isinstance(error, ResponseError):
            greeks_output.append_stdout(f"Request ID: {error.request_id}\n")
        greeks_status.value = "Request failed; no automatic retry."
    except ValueError as error:
        greeks_output.append_stdout(str(error) + "\n")
        greeks_status.value = "Check the option inputs."
    finally:
        set_busy(False)


def solve_iv(_):
    global last_iv_response, iv_snapshot
    set_busy(True)
    iv_status.value = "Solving implied volatility…"
    last_iv_response = iv_snapshot = None
    iv_output.outputs = ()
    try:
        request = current_iv_request()
        with MorphIQ() as client:
            response = tracked_request("Solve IV", client.implied_volatility, request)
        last_iv_response = response
        result = response.result
        iv_snapshot = {
            "Underlying source": spot_source,
            **request.to_wire(),
            "Requested model": request.pricing_model,
            **result.model_dump(),
            "CU charged": response.metadata.fcu_charged
            if response.metadata.fcu_charged is not None
            else "not reported",
            "Request ID": response.metadata.request_id,
        }
        with iv_output.hold_sync():
            iv_output.append_stdout(
                f"Implied volatility: {result.implied_volatility * 100:.12g}% "
                f"({result.implied_volatility} decimal).\n"
                f"Iterations: {result.iterations}; used fallback: {result.used_fallback}.\n"
                f"Residual: {result.residual}; basis: {result.residual_basis}.\n"
                "The pricing volatility assumption is unchanged.\n"
            )
            iv_output.append_display_data(pd.DataFrame([iv_snapshot]))
            iv_output.append_stdout(
                "Engines: "
                + "; ".join(
                    f"{e.name} {e.version} ({e.revision})" for e in response.metadata.engines
                )
                + "\n"
            )
        iv_status.value = "IV complete. Results retain the inputs shown below."
    except APIError as error:
        if isinstance(error, RateLimitError):
            iv_output.append_stdout(
                "Rate limited (429). No retry was made. Retry-After: "
                + (error.retry_after if error.retry_after is not None else "not supplied")
                + "\n"
            )
        charge = error.metadata.fcu_charged if error.metadata else None
        iv_output.append_stdout(
            f"HTTP {error.status_code}: {error.code}: {error.message}\n"
            f"Request ID: {error.request_id}; "
            f"CU charged: {charge if charge is not None else 'not reported'}\n"
        )
        iv_status.value = "Request failed; no automatic retry."
    except (TransportError, ResponseError) as error:
        iv_output.append_stdout(
            "No usable response. Execution and charges may be unknown; no retry was made.\n"
        )
        if isinstance(error, ResponseError):
            iv_output.append_stdout(f"Request ID: {error.request_id}\n")
        iv_status.value = "Request failed; no automatic retry."
    except ValueError as error:
        iv_output.append_stdout(str(error) + "\n")
        iv_status.value = "Check the option inputs."
    finally:
        set_busy(False)


def analyze_premium(_):
    global last_analytics_response, analytics_snapshot, analytics_table
    set_busy(True)
    analytics_status.value = "Solving IV and calculating price and Greeks…"
    last_analytics_response = analytics_snapshot = analytics_table = None
    analytics_output.outputs = ()
    try:
        # Volatility is a required wire field, but the service ignores it with a premium.
        request = AnalyticsRequest(
            **current_iv_request().to_wire(), volatility=volatility.value / 100
        )
        with MorphIQ() as client:
            response = tracked_request("Analyze premium", client.analytics, request)
        last_analytics_response = response
        solved = response.result.implied_volatility
        analytics_snapshot = {
            "Underlying source": spot_source,
            **request.to_wire(),
            "Requested model": request.pricing_model,
            "Model": response.result.pricing_model,
            "Version": response.result.model_version,
            "Price": response.result.price,
            "CU charged": response.metadata.fcu_charged
            if response.metadata.fcu_charged is not None
            else "not reported",
            "Request ID": response.metadata.request_id,
        }
        units = {
            "delta": "dV/dS",
            "gamma": "d²V/dS²",
            "theta": "dV/dt · per year",
            "vega": "dV/dσ · per unit volatility",
            "rho": "dV/dr · per unit rate",
            "vanna": "d²V/(dS dσ) · per unit volatility",
            "volga": "d²V/dσ² · per unit volatility squared",
            "charm": "d²V/(dS dt) · per year",
            "veta": "d²V/(dσ dt) · per year per unit volatility",
            "color": "d³V/(dS² dt) · per year",
        }
        analytics_table = pd.DataFrame(
            [
                {"Greek": name, "Value": getattr(response.result.greeks, name), "Basis": basis}
                for name, basis in units.items()
            ]
        ).set_index("Greek")
        with analytics_output.hold_sync():
            analytics_output.append_stdout(
                f"Option value: {response.result.price:.12g} per share.\n"
            )
            if solved is not None:
                analytics_snapshot["Solved volatility"] = solved.implied_volatility
                analytics_output.append_stdout(
                    f"Implied volatility: {solved.implied_volatility * 100:.12g}% "
                    f"({solved.implied_volatility} decimal).\n"
                    f"Iterations: {solved.iterations}; used fallback: {solved.used_fallback}.\n"
                    f"Residual: {solved.residual}; basis: {solved.residual_basis}.\n"
                    "Greeks use the solved volatility. Your pricing assumption is unchanged.\n"
                )
            else:
                analytics_output.append_stdout(
                    "The service did not report an IV solve; "
                    "the volatility used for these Greeks is unconfirmed.\n"
                )
            analytics_output.append_display_data(analytics_table.style.format({"Value": "{:.12g}"}))
            analytics_output.append_display_data(pd.DataFrame([analytics_snapshot]))
            analytics_output.append_stdout(
                "Engines: "
                + "; ".join(
                    f"{e.name} {e.version} ({e.revision})" for e in response.metadata.engines
                )
                + "\n"
            )
        analytics_status.value = (
            "Analysis complete. The combined request is counted once in session CU."
        )
    except APIError as error:
        if isinstance(error, RateLimitError):
            analytics_output.append_stdout(
                "Rate limited (429). No retry was made. Retry-After: "
                + (error.retry_after if error.retry_after is not None else "not supplied")
                + "\n"
            )
        charge = error.metadata.fcu_charged if error.metadata else None
        analytics_output.append_stdout(
            f"HTTP {error.status_code}: {error.code}: {error.message}\n"
            f"Request ID: {error.request_id}; "
            f"CU charged: {charge if charge is not None else 'not reported'}\n"
        )
        analytics_status.value = "Request failed; no automatic retry."
    except (TransportError, ResponseError) as error:
        analytics_output.append_stdout(
            "No usable response. Execution and charges may be unknown; no retry was made.\n"
        )
        if isinstance(error, ResponseError):
            analytics_output.append_stdout(f"Request ID: {error.request_id}\n")
        analytics_status.value = "Request failed; no automatic retry."
    except ValueError as error:
        analytics_output.append_stdout(str(error) + "\n")
        analytics_status.value = "Check the option inputs."
    finally:
        set_busy(False)


def sweep_spot(_):
    global sweep_results
    sweep_results = []
    sweep_output.outputs = ()
    set_busy(True)
    try:
        base = current_request()
        requests = [
            PriceRequest(**{**base.to_wire(), "spot": base.spot * (1 + step * 0.05)})
            for step in range(-4, 5)
        ]
        with MorphIQ() as client:
            response = tracked_request(
                "Spot sensitivity", client.price_batch, BatchPriceRequest(requests=requests)
            )
        for item in response.results:
            request = requests[item.index]
            row = {
                "Index": item.index,
                "Spot": request.spot,
                "Strike": request.strike,
                "Style": request.exercise_style,
                "Type": request.option_type,
                "Years": request.time_to_expiry_years,
                "Vol (%)": request.volatility * 100,
                "Rate (%)": request.risk_free_rate * 100,
                "Yield (%)": request.dividend_yield * 100,
                "Status": item.status,
                "Price": None,
                "Error": None,
            }
            if item.status == "success":
                row.update(
                    Price=item.result.price,
                    Model=item.result.pricing_model,
                    Version=item.result.model_version,
                )
            else:
                row.update(
                    Error=item.error.code, Message=item.error.message, Field=item.error.field
                )
            sweep_results.append(row)
        table = pd.DataFrame(sweep_results)
        charge = response.metadata.fcu_charged
        with sweep_output.hold_sync():
            sweep_output.append_stdout(
                f"One batch, {len(sweep_results)} outcomes. "
                f"Request ID: {response.metadata.request_id}. "
                f"Batch CU charged: {charge if charge is not None else 'not reported'}.\n"
                "Charge is counted once in Session CU; no per-item charge is inferred.\n"
            )
            sweep_output.append_display_data(table)
            fig, ax = plt.subplots(figsize=(8, 3.5))
            # Failed points remain gaps, never zero-valued prices or bridged lines.
            ax.plot(
                table["Spot"],
                [r["Price"] if r["Price"] is not None else float("nan") for r in sweep_results],
                marker="o",
                color="#2459a6",
            )
            ax.set(
                xlabel="Underlying price",
                ylabel="Option value per share",
                title=f"{base.exercise_style.title()} {base.option_type}: spot sensitivity",
            )
            ax.grid(alpha=0.25)
            sweep_output.append_display_data(fig)
            plt.close(fig)
    except (APIError, TransportError, ResponseError, ValueError) as error:
        # Append a complete failure report to the widget on the first click.
        if isinstance(error, APIError):
            charge = error.metadata.fcu_charged if error.metadata else None
            sweep_output.append_stdout(
                f"HTTP {error.status_code}: {error.code}: {error.message}\n"
                f"Request ID: {error.request_id}; "
                f"CU charged: {charge if charge is not None else 'not reported'}\n"
            )
            if isinstance(error, RateLimitError):
                sweep_output.append_stdout(
                    "Rate limited. Retry-After: "
                    + (error.retry_after if error.retry_after is not None else "not supplied")
                    + "\n"
                )
        elif isinstance(error, (TransportError, ResponseError)):
            sweep_output.append_stdout(
                "No usable batch response; execution and charges may be unknown.\n"
                f"Request ID: {getattr(error, 'request_id', None)}\n"
            )
        else:
            sweep_output.append_stdout(str(error) + "\n")
        sweep_output.append_stdout("No retry or batch splitting was performed. See Session CU.\n")
    finally:
        set_busy(False)


price_button.on_click(price_selected)
greeks_button.on_click(calculate_greeks)
iv_button.on_click(solve_iv)
analytics_button.on_click(analyze_premium)
show_usage()
sweep_button.on_click(sweep_spot)
display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Underlying</h3>"),
            widgets.HBox([ticker, price_interval], layout=widgets.Layout(flex_flow="row wrap")),
            market_output,
            widgets.HTML("<h3>Option and assumptions</h3>"),
            widgets.HBox([exercise, option_type]),
            model,
            widgets.HBox([spot, strike]),
            widgets.HBox([expiry, volatility]),
            widgets.HBox([rate, yield_input]),
            widgets.HTML(
                "<p>Rates and volatility are percentages. Expiry is in years. "
                "Uses continuous dividend yield; cash-dividend schedules are not entered here.</p>"
            ),
            widgets.HBox(
                [price_button, greeks_button, sweep_button],
                layout=widgets.Layout(flex_flow="row wrap"),
            ),
            result_output,
            greeks_status,
            greeks_output,
            sweep_output,
            widgets.HTML(
                "<h3>Implied volatility</h3><p>Enter the market option premium "
                "per share, in the same currency as spot and strike. "
                "Solve IV uses the option inputs above, excluding volatility.</p>"
            ),
            widgets.HBox(
                [market_price, iv_button, analytics_button],
                layout=widgets.Layout(flex_flow="row wrap"),
            ),
            iv_status,
            iv_output,
            analytics_status,
            analytics_output,
            widgets.HTML(
                "<h3>Session CU</h3><p>Reported charges for every request in this "
                "pricing workspace, including failed requests. One combined response "
                "counts once. Unknown charges make the total incomplete. "
                "This is a kernel-session tally, not your account balance.</p>"
            ),
            usage_summary,
            usage_output,
        ]
    )
)

In [ ]:
ticker.observe(load_close, names="value")
price_interval.observe(load_close, names="value")
load_close(None)

### Reading the analysis

Prices are per share in the same currency as the underlying and strike; no contract
multiplier is applied. Daily mode uses the last completed close. Intraday mode uses the latest returned
regular-session bar and displays its timestamp and timezone; the bar may still be
forming and data may be delayed.
Volatility, rates and continuous dividend yield are your assumptions. Enter 20 for 20%.
Expiry is expressed in years. Cash-dividend schedules are outside these controls.

The Greeks table labels each derivative. Theta, charm, veta and color use annual
time units; vega and rho are per unit volatility and rate, not per percentage point.
The snapshot below the table records the exact option inputs, model, request ID and CU.

IV solves the volatility implied by your entered market premium under the selected
model. It does not fetch an option quote or replace your volatility assumption.
The result includes the residual and its basis: `forward_normalised_price` or
`option_price`. Residuals on different bases are not directly comparable.
`used_fallback` describes the algorithm, not solution quality; American bisection
reports it as true. Iterations and diagnostics are preserved as reported by the API.

The sensitivity chart contains separate model valuations, not forecasts or quoted
market option prices. Each row records the inputs, resolved model, request ID and
reported commercial CU. A missing charge is unreported, not zero. If a request fails,
per-item errors remain visible alongside successes. Failed chart points are gaps;
no request or failed item is retried automatically.

Results retain their original inputs when controls change. Clear outputs before sharing;
shut down the kernel when finished. Source code is available by expanding the hidden cell.

Session CU sums only `fcu_charged`, once per actual request. It includes reported
charges on errors and mixed-outcome batches. Zero is counted as zero; missing charges and
timeouts remain unknown. The displayed known subtotal is not a complete total
while any charges are unknown. Quotes and internal compute units are not added.
Rerunning setup retains the ledger; restarting the kernel starts a new session.
